# My task write-up: Pole / geometry evaluation tooling

**Stream:** ML & Deployment
**What this notebook is:** study notes for the reusable size / aspect-ratio evaluation tool I added on top of the existing WalkBuddy eval pipeline. I wrote it after the tool was implemented and after I ran the tests myself.

I am not claiming this is a new detector, a new matching algorithm, or a held-out test result. It is a breakdown tool. It slices the same detections the old evaluator already scores, and it is only allowed to run on train / val evidence.

## 1. The task

I was asked to turn the existing evaluation into a **reusable tool** that reports detection performance broken down by object **size** and **aspect ratio / geometry**. The pole class was the reason for the task — poles are thin, easy to miss, and the overall per-class number hides that — but the tool had to work for any class, not just pole.



- report class performance by bbox-area size buckets (small / medium / large)
- report performance by geometry / aspect ratio (tall-thin poles and so on)
- stay reusable for any class, with a pole-focused view when pole is in the class list
- machine-readable output (JSON and CSV) plus a clear human-readable summary
- automated tests
- use train / validation / evaluation evidence only — **do not tune against the held-out test set**
- document how to rerun this for Candidate 2 and later models

I was also told to study the existing eval stack first and **build on it**, not duplicate IoU matching.

## 2. What already existed (so I didn't duplicate it)

Before writing anything I read `ML_side/evaluation/metrics.py`, `matching.py`, `run_eval.py`, `report.py`, and the small JSON fixtures under `tests/fixtures/eval/`.

What I found:

- The repo already had a working evaluator: `evaluation.metrics.evaluate()`. It takes ground-truth and prediction JSON in a simple shape (`image_id` + `boxes` with `class`, `bbox` as `[x_min, y_min, x_max, y_max]`, and `score` on predictions). It matches boxes **per class per image** with greedy IoU (`match_class_detections` in `matching.py`) and returns per-class precision / recall / F1 plus missed-hazard and false-detection lists.
- `run_eval.py` is the CLI for that overall error-analysis report (JSON + Markdown). Same JSON inputs.
- There is a *different* tool, `tools/evaluate_current_model.py`, that wraps Ultralytics `model.val()` and reports mAP. That one cannot slice individual boxes by size or aspect, so I did not try to extend it.
- The existing fixture pole is already a tall-thin box: `[500, 100, 520, 470]` (20×370 px). So the old pipeline already knew how to score a pole — it just reported one number for the whole class.

I decided **not to touch `matching.py` or `evaluate()`**. Reimplementing IoU would have been a second source of truth, and the tests for the old pipeline would no longer prove that my buckets use the same matching rules.

What I added instead is a slice layer: filter the same records into a bucket, then call the existing `evaluate()` on that subset. Ground-truth and predictions are filtered independently (COCO-style). I will come back to what that means for FN/FP in the design section.

The code cell below is the existing `evaluate()` function from `metrics.py` — code I studied, not code I wrote.

In [ ]:
# --- EXISTING CODE I STUDIED (not mine) ---
# From ML_side/evaluation/metrics.py — evaluate() as it already existed.
# I did not change this function. My tool calls it per bucket.

def evaluate(
    ground_truth: List[dict],
    predictions: List[dict],
    classes: Optional[Sequence[str]] = None,
    iou_threshold: float = 0.5,
    severity_map: Optional[dict] = None,
    strict: bool = True,
) -> dict:
    """Run the full evaluation and return one deterministic result dict.

    Given the same ground_truth, predictions, classes and iou_threshold,
    this always returns the same result, there is no randomness and no
    dependence on dict/set iteration order (image ids and classes are
    always processed in a fixed, sorted order).

    strict=True (the default) raises TaxonomyMismatchError if any box in
    either input references a class that doesn't canonicalize to an
    approved class, this is meant to catch a real model/taxonomy mismatch
    rather than hide it. strict=False instead includes those occurrences
    in the result under "unknown_classes" and proceeds using only the
    recognized boxes.
    """
    classes = list(classes) if classes is not None else list(TAXONOMY_CLASSES)
    severity_map = severity_map if severity_map is not None else DEFAULT_SEVERITY

    gt_index, gt_unknown = _index_by_image_and_class(
        ground_truth, classes, TAXONOMY_CLASSES, "ground_truth"
    )
    pred_index, pred_unknown = _index_by_image_and_class(
        predictions, classes, TAXONOMY_CLASSES, "prediction"
    )
    unknown_classes = gt_unknown + pred_unknown
    unknown_classes.sort(key=lambda u: (u["image_id"], u["source"], str(u["class"])))

    if strict and unknown_classes:
        names = sorted({str(u["class"]) for u in unknown_classes})
        raise TaxonomyMismatchError(
            "Found box(es) with a class that isn't in the approved taxonomy: "
            f"{', '.join(names)}. Pass strict=False to surface these in the "
            "report instead of failing."
        )

    # Ground truth defines the evaluated set of images. A prediction given
    # for an image_id with no matching ground truth entry is almost always a
    # data problem (a typo'd or mismatched id), not a real result, including
    # it would fabricate false_detections against an image that was never
    # actually part of this evaluation. Those ids are excluded from scoring
    # and surfaced separately instead of silently treated as real data.
    gt_ids = set(gt_index)
    pred_ids = set(pred_index)
    predictions_without_ground_truth = sorted(pred_ids - gt_ids)
    image_ids = sorted(gt_ids)

    counts = {c: {"tp": 0, "fp": 0, "fn": 0, "support": 0} for c in classes}
    missed_hazards = []
    false_detections = []

    for image_id in image_ids:
        for cls in classes:
            gt_boxes = gt_index[image_id].get(cls, [])
            pred_boxes = pred_index[image_id].get(cls, [])
            counts[cls]["support"] += len(gt_boxes)

            matches, fp_idx, fn_idx = match_class_detections(gt_boxes, pred_boxes, iou_threshold)
            counts[cls]["tp"] += len(matches)
            counts[cls]["fp"] += len(fp_idx)
            counts[cls]["fn"] += len(fn_idx)

            for gi in fn_idx:
                gt = gt_boxes[gi]
                missed_hazards.append({
                    "image_id": image_id,
                    "class": cls,
                    "bbox": gt["bbox"],
                    "severity": severity_map.get(cls, "UNKNOWN"),
                })
            for pi in fp_idx:
                pred = pred_boxes[pi]
                false_detections.append({
                    "image_id": image_id,
                    "class": cls,
                    "bbox": pred["bbox"],
                    "score": pred["score"],
                })

    per_class = {}
    for c in classes:
        tp, fp, fn = counts[c]["tp"], counts[c]["fp"], counts[c]["fn"]
        precision, recall, f1 = _precision_recall_f1(tp, fp, fn)
        per_class[c] = {
            "tp": tp, "fp": fp, "fn": fn,
            "support": counts[c]["support"],
            "precision": precision, "recall": recall, "f1": f1,
        }

    micro_tp = sum(counts[c]["tp"] for c in classes)
    micro_fp = sum(counts[c]["fp"] for c in classes)
    micro_fn = sum(counts[c]["fn"] for c in classes)
    micro_p, micro_r, micro_f1 = _precision_recall_f1(micro_tp, micro_fp, micro_fn)

    classes_with_signal = [
        c for c in classes
        if counts[c]["support"] > 0 or (counts[c]["tp"] + counts[c]["fp"]) > 0
    ]
    if classes_with_signal:
        macro_p = sum(per_class[c]["precision"] or 0.0 for c in classes_with_signal) / len(classes_with_signal)
        macro_r = sum(per_class[c]["recall"] or 0.0 for c in classes_with_signal) / len(classes_with_signal)
        macro_f1 = sum(per_class[c]["f1"] or 0.0 for c in classes_with_signal) / len(classes_with_signal)
    else:
        macro_p = macro_r = macro_f1 = None

    def _severity_sort_key(severity_name: str) -> int:
        # severity_rank() returns higher numbers for more severe entries
        # (CRITICAL=4 ... LOW=1); negate so CRITICAL sorts first. Anything
        # that isn't a recognized BaseSeverity name (e.g. a caller-supplied
        # severity_map with a typo) sorts last instead of raising.
        try:
            return -severity_rank(BaseSeverity[severity_name])
        except KeyError:
            return 1

    missed_hazards.sort(key=lambda h: (_severity_sort_key(h["severity"]), h["image_id"], h["class"]))
    false_detections.sort(key=lambda d: (d["image_id"], d["class"]))

    return {
        "config": {"iou_threshold": iou_threshold, "classes": classes, "strict": strict},
        "num_images": len(image_ids),
        "per_class": per_class,
        "overall": {
            "micro": {"tp": micro_tp, "fp": micro_fp, "fn": micro_fn,
                      "precision": micro_p, "recall": micro_r, "f1": micro_f1},
            "macro": {"precision": macro_p, "recall": macro_r, "f1": macro_f1},
        },
        "missed_hazards": missed_hazards,
        "false_detections": false_detections,
        "unknown_classes": unknown_classes,
        "unmatched_image_ids": {
            "predictions_without_ground_truth": predictions_without_ground_truth,
        },
    }


## 3. Files I added

I added four Python modules (plus synthetic fixtures and a docs section). I kept them next to the existing eval package so they consume the same JSON and the same `evaluate()`:

| File | Role |
|---|---|
| `evaluation/geometry.py` | size / aspect bucketing + `evaluate_by_geometry()` |
| `evaluation/geometry_report.py` | JSON (source of truth), CSV, Markdown |
| `evaluation/run_geometry_eval.py` | CLI sibling of `run_eval.py` |
| `tests/test_evaluation_geometry.py` | tests + `tests/fixtures/eval/geometry/` |

I did not change `matching.py`, `metrics.py`, or `run_eval.py`.

### 3.1 `evaluation/geometry.py` — why and what

This is the core. I needed somewhere to:

- compute bbox **area** and **height/width**
- assign a size bucket (COCO 32² / 96² by default, but configurable)
- assign an aspect bucket (`tall_thin` ≥ 3.0, `tall` ≥ 1.5, `square_ish` ≥ 2/3, `wide` below that)
- `filter_records()` so every `image_id` stays in the list (empty images still count) while boxes that are not in the bucket are dropped
- `evaluate_by_geometry()` which calls existing `evaluate()` once unstratified, once per size bucket, once per aspect bucket, and once per size×aspect pair
- a **hard held-out guard**: `--split` is required; `test` / `held-out` always raise `HeldOutEvidenceError`; reports always record `held_out_test_used: false`

I also refuse inputs where every box area is ≤ 1.0, because that almost certainly means someone passed normalized YOLO boxes, which would all fall into `small` under pixel² COCO cuts.

The next cell is the **actual file**.

In [ ]:
"""Size and aspect-ratio bucketing on top of evaluate().

This module does not reimplement IoU matching, precision, or recall.
It slices the same ground-truth / prediction JSON that evaluate() already
consumes, then calls evaluate() once per bucket.

Ground-truth and predictions are filtered independently (COCO-style). A
medium ground-truth box whose matching prediction is large is therefore a
false negative in the medium bucket and a false positive in the large
bucket.

Coordinates are pixel xyxy, the same unit load_yolo_predict_fn() writes
and the existing eval fixtures use. COCO area thresholds are in px².
"""

from typing import Callable, List, Optional, Sequence

from .metrics import evaluate
from .taxonomy import TAXONOMY_CLASSES, canonicalize_class_name


DEFAULT_SMALL_AREA_MAX = float(32 ** 2)  # 1024, exclusive upper bound
DEFAULT_MEDIUM_AREA_MAX = float(96 ** 2)  # 9216, exclusive upper bound
DEFAULT_TALL_THIN_MIN = 3.0
DEFAULT_TALL_MIN = 1.5
DEFAULT_SQUARE_MIN = 2.0 / 3.0

SIZE_BUCKET_ORDER = ("small", "medium", "large")
ASPECT_BUCKET_ORDER = ("tall_thin", "tall", "square_ish", "wide")

ALLOWED_SPLITS = ("train", "val")
SPLIT_ALIASES = {
    "train": "train",
    "val": "val",
    "validation": "val",
    "eval": "val",
}
FORBIDDEN_SPLIT_TOKENS = ("test", "heldout", "held-out", "held_out")

COORDINATE_SPACE = "pixel_xyxy"


class HeldOutEvidenceError(ValueError):
    """Raised when a held-out test split or path is supplied. No override exists."""


class SizeThresholds:
    """COCO-style area cuts. small_max and medium_max are exclusive upper bounds."""

    def __init__(
        self,
        small_max: float = DEFAULT_SMALL_AREA_MAX,
        medium_max: float = DEFAULT_MEDIUM_AREA_MAX,
    ):
        if not (0 < small_max < medium_max):
            raise ValueError(
                f"Size thresholds must satisfy 0 < small_max < medium_max; "
                f"got small_max={small_max}, medium_max={medium_max}."
            )
        self.small_max = float(small_max)
        self.medium_max = float(medium_max)

    def as_edges(self) -> dict:
        return {
            "small": [0, self.small_max],
            "medium": [self.small_max, self.medium_max],
            "large": [self.medium_max, None],
        }


class AspectThresholds:
    """Height/width cuts. Lower bounds are inclusive, upper bounds exclusive."""

    def __init__(
        self,
        tall_thin_min: float = DEFAULT_TALL_THIN_MIN,
        tall_min: float = DEFAULT_TALL_MIN,
        square_min: float = DEFAULT_SQUARE_MIN,
    ):
        if not (0 < square_min < tall_min < tall_thin_min):
            raise ValueError(
                "Aspect thresholds must satisfy 0 < square_min < tall_min < "
                f"tall_thin_min; got square_min={square_min}, tall_min={tall_min}, "
                f"tall_thin_min={tall_thin_min}."
            )
        self.tall_thin_min = float(tall_thin_min)
        self.tall_min = float(tall_min)
        self.square_min = float(square_min)

    def as_edges(self) -> dict:
        return {
            "tall_thin": {"min_hw": self.tall_thin_min, "max_hw": None},
            "tall": {"min_hw": self.tall_min, "max_hw": self.tall_thin_min},
            "square_ish": {"min_hw": self.square_min, "max_hw": self.tall_min},
            "wide": {"min_hw": None, "max_hw": self.square_min},
        }


def bbox_area(bbox: Sequence[float]) -> float:
    """Pixel area of an [x_min, y_min, x_max, y_max] box; 0 if inverted or empty."""
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    if width <= 0 or height <= 0:
        return 0.0
    return width * height


def bbox_aspect_hw(bbox: Sequence[float]) -> Optional[float]:
    """Height / width, or None when the box has no positive width/height."""
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    if width <= 0 or height <= 0:
        return None
    return height / width


def _box_bbox(box: dict) -> Optional[list]:
    bbox = box.get("bbox")
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        return None
    return list(bbox)


def size_bucket_for_area(area: float, thresholds: SizeThresholds) -> str:
    if area is None or area <= 0:
        return "invalid"
    if area < thresholds.small_max:
        return "small"
    if area < thresholds.medium_max:
        return "medium"
    return "large"


def aspect_bucket_for_hw(hw: Optional[float], thresholds: AspectThresholds) -> str:
    if hw is None:
        return "invalid"
    if hw >= thresholds.tall_thin_min:
        return "tall_thin"
    if hw >= thresholds.tall_min:
        return "tall"
    if hw >= thresholds.square_min:
        return "square_ish"
    return "wide"


def box_size_bucket(box: dict, thresholds: SizeThresholds) -> str:
    bbox = _box_bbox(box)
    if bbox is None:
        return "invalid"
    return size_bucket_for_area(bbox_area(bbox), thresholds)


def box_aspect_bucket(box: dict, thresholds: AspectThresholds) -> str:
    bbox = _box_bbox(box)
    if bbox is None:
        return "invalid"
    return aspect_bucket_for_hw(bbox_aspect_hw(bbox), thresholds)


def filter_records(records: List[dict], predicate: Callable[[dict], bool]) -> List[dict]:
    """Keep every image_id; keep only boxes for which predicate(box) is True.

    Extra per-record fields (for example image_path) are preserved. The
    original records are not mutated.
    """
    filtered = []
    for rec in records:
        new_rec = dict(rec)
        new_rec["boxes"] = [box for box in rec.get("boxes", []) if predicate(box)]
        filtered.append(new_rec)
    return filtered


def _normalize_held_out_token(value: str) -> str:
    return value.strip().lower().replace("_", "-")


def _component_is_test(part: str) -> bool:
    name = part.lower()
    if name == "test":
        return True
    if "." in name:
        stem = name.rsplit(".", 1)[0]
        if stem == "test":
            return True
    return False


def _component_is_held_out(part: str) -> bool:
    compact = part.lower().replace("_", "").replace("-", "")
    if "heldout" in compact:
        return True
    return _normalize_held_out_token(part) in {"held-out", "heldout"}


def path_is_held_out(path) -> bool:
    """True when a filesystem path names the held-out test split.

    A path component exactly named ``test`` (or a file stem ``test``) is
    refused. ``tests`` is not, so this package's own pytest fixtures remain
    usable. Any component containing held-out / heldout / held_out is refused.
    """
    from pathlib import Path

    for part in Path(path).parts:
        if _component_is_test(part) or _component_is_held_out(part):
            return True
    return False


def refuse_held_out_path(path) -> None:
    if path_is_held_out(path):
        raise HeldOutEvidenceError(
            f"Held-out test evidence is not allowed: {path}. "
            "Use train or val (aliases: validation, eval) only."
        )


def canonical_split(split: str) -> str:
    """Map an allowed split name to train/val. Held-out names always fail."""
    if split is None or str(split).strip() == "":
        raise HeldOutEvidenceError(
            "dataset split is required; allowed values are train, val "
            "(aliases: validation, eval). The held-out test split is not allowed."
        )
    key = str(split).strip().lower()
    compact = key.replace("_", "").replace("-", "")
    if key in {"test"} or compact in {"test", "heldout", "heldouttest"} or "heldout" in compact:
        raise HeldOutEvidenceError(
            "The held-out test split cannot be used. "
            "Use train or val (aliases: validation, eval)."
        )
    if key in SPLIT_ALIASES:
        return SPLIT_ALIASES[key]
    allowed = ", ".join(ALLOWED_SPLITS) + " (aliases: validation, eval)"
    raise ValueError(f"Unknown dataset split {split!r}. Allowed: {allowed}.")


def refuse_held_out_inputs(ground_truth_path, predictions_path, split: str) -> str:
    """Validate split and input paths. Returns the canonical split name."""
    canonical = canonical_split(split)
    refuse_held_out_path(ground_truth_path)
    refuse_held_out_path(predictions_path)
    return canonical


def assert_pixel_xyxy(ground_truth: List[dict], predictions: List[dict]) -> None:
    """Refuse inputs that look like normalized YOLO boxes, not pixel xyxy."""
    areas = []
    for records in (ground_truth, predictions):
        for rec in records:
            for box in rec.get("boxes", []):
                bbox = _box_bbox(box)
                if bbox is None:
                    continue
                area = bbox_area(bbox)
                if area > 0:
                    areas.append(area)
    if areas and max(areas) <= 1.0:
        raise ValueError(
            "All bounding-box areas are <= 1.0; geometry evaluation expects "
            "pixel xyxy coordinates because COCO size buckets are in pixel^2. "
            "Normalized YOLO boxes would all fall into the small bucket."
        )


def _canonical_box_class(box: dict, classes: Sequence[str]) -> Optional[str]:
    raw = box.get("class")
    if raw is None:
        return None
    canonical = canonicalize_class_name(raw)
    if canonical is None or canonical not in classes:
        return None
    return canonical


def _bucket_support(
    ground_truth: List[dict],
    classes: Sequence[str],
    size_thresholds: SizeThresholds,
    aspect_thresholds: AspectThresholds,
) -> dict:
    size_counts = {
        c: {bucket: 0 for bucket in SIZE_BUCKET_ORDER + ("invalid",)} for c in classes
    }
    aspect_counts = {
        c: {bucket: 0 for bucket in ASPECT_BUCKET_ORDER + ("invalid",)} for c in classes
    }
    for rec in ground_truth:
        for box in rec.get("boxes", []):
            cls = _canonical_box_class(box, classes)
            if cls is None:
                continue
            size_counts[cls][box_size_bucket(box, size_thresholds)] += 1
            aspect_counts[cls][box_aspect_bucket(box, aspect_thresholds)] += 1
    return {"size": size_counts, "aspect": aspect_counts}


def _metrics_view(result: dict) -> dict:
    return {
        "num_images": result["num_images"],
        "per_class": result["per_class"],
        "overall": result["overall"],
        "missed_hazards": result["missed_hazards"],
        "false_detections": result["false_detections"],
    }


def _class_metrics(result: dict, cls: str) -> dict:
    return dict(result["per_class"][cls])


def evaluate_by_geometry(
    ground_truth: List[dict],
    predictions: List[dict],
    *,
    split: str,
    classes: Optional[Sequence[str]] = None,
    iou_threshold: float = 0.5,
    size_thresholds: Optional[SizeThresholds] = None,
    aspect_thresholds: Optional[AspectThresholds] = None,
    strict: bool = True,
) -> dict:
    """Score detections overall and inside each size / aspect bucket.

    ``split`` is required. Held-out names (test, held-out, …) always raise
    HeldOutEvidenceError; there is no override flag.
    """
    canonical = canonical_split(split)
    classes = list(classes) if classes is not None else list(TAXONOMY_CLASSES)
    size_thresholds = size_thresholds if size_thresholds is not None else SizeThresholds()
    aspect_thresholds = (
        aspect_thresholds if aspect_thresholds is not None else AspectThresholds()
    )

    assert_pixel_xyxy(ground_truth, predictions)

    unstratified = evaluate(
        ground_truth,
        predictions,
        classes=classes,
        iou_threshold=iou_threshold,
        strict=strict,
    )

    by_size = {}
    for bucket in SIZE_BUCKET_ORDER:
        gt_b = filter_records(
            ground_truth, lambda box, b=bucket: box_size_bucket(box, size_thresholds) == b
        )
        pred_b = filter_records(
            predictions, lambda box, b=bucket: box_size_bucket(box, size_thresholds) == b
        )
        by_size[bucket] = _metrics_view(
            evaluate(
                gt_b,
                pred_b,
                classes=classes,
                iou_threshold=iou_threshold,
                strict=strict,
            )
        )

    by_aspect = {}
    for bucket in ASPECT_BUCKET_ORDER:
        gt_b = filter_records(
            ground_truth, lambda box, b=bucket: box_aspect_bucket(box, aspect_thresholds) == b
        )
        pred_b = filter_records(
            predictions, lambda box, b=bucket: box_aspect_bucket(box, aspect_thresholds) == b
        )
        by_aspect[bucket] = _metrics_view(
            evaluate(
                gt_b,
                pred_b,
                classes=classes,
                iou_threshold=iou_threshold,
                strict=strict,
            )
        )

    by_size_and_aspect = {cls: {} for cls in classes}
    for size_bucket in SIZE_BUCKET_ORDER:
        for aspect_bucket in ASPECT_BUCKET_ORDER:
            key = f"{size_bucket}|{aspect_bucket}"

            def _both(box, s=size_bucket, a=aspect_bucket):
                return (
                    box_size_bucket(box, size_thresholds) == s
                    and box_aspect_bucket(box, aspect_thresholds) == a
                )

            crossed = evaluate(
                filter_records(ground_truth, _both),
                filter_records(predictions, _both),
                classes=classes,
                iou_threshold=iou_threshold,
                strict=strict,
            )
            for cls in classes:
                by_size_and_aspect[cls][key] = _class_metrics(crossed, cls)

    return {
        "config": {
            "iou_threshold": iou_threshold,
            "classes": classes,
            "strict": strict,
            "dataset_split": canonical,
            "held_out_test_used": False,
            "coordinate_space": COORDINATE_SPACE,
            "size_buckets": size_thresholds.as_edges(),
            "aspect_buckets": aspect_thresholds.as_edges(),
        },
        "num_images": unstratified["num_images"],
        "unstratified": _metrics_view(unstratified),
        "by_size": by_size,
        "by_aspect": by_aspect,
        "by_size_and_aspect": by_size_and_aspect,
        "bucket_support": _bucket_support(
            ground_truth, classes, size_thresholds, aspect_thresholds
        ),
        "unknown_classes": unstratified["unknown_classes"],
        "unmatched_image_ids": unstratified["unmatched_image_ids"],
    }


### 3.2 `evaluation/geometry_report.py` — why and what

`report.py` already builds the overall error-analysis JSON/Markdown, and it is tied to `artifact_type: supplementary_error_analysis`. I did not want to overload that file, so I added a separate reporter for `artifact_type: geometry_breakdown`.

What I wanted:

- **JSON** as the source of truth (same determinism trick as the old reporter: `generated_at` is passed in, not read from the clock inside the builder)
- **CSV** as a flat table: one row per dimension × bucket × class, so someone can open it in a spreadsheet without parsing nested JSON
- **Markdown** a person can read. If `pole` is in the requested class list, the **Pole focus** section is printed first (size, aspect, then size×aspect). If you pass `--classes table` only, that section is omitted on purpose — the tool is reusable, not pole-only.

I also put the independent-filtering explanation into the Markdown so a reviewer of a report does not have to go read the source to understand why unstratified TP can become FN+FP across size buckets.

Actual file:

In [ ]:
"""JSON, CSV, and Markdown reports for evaluate_by_geometry() results.

Builders are pure functions of their inputs. generated_at is an optional
argument rather than read from the clock, so tests can assert exact output.
JSON is the source of truth; CSV and Markdown are projections of the same dict.
"""

import csv
import json
from pathlib import Path
from typing import Optional, Union

from .geometry import ASPECT_BUCKET_ORDER, SIZE_BUCKET_ORDER


ARTIFACT_TYPE = "geometry_breakdown"
SCHEMA_VERSION = "1.0.0"

CSV_FIELDNAMES = (
    "dimension",
    "bucket",
    "class",
    "support",
    "tp",
    "fp",
    "fn",
    "precision",
    "recall",
    "f1",
)

INDEPENDENT_FILTERING_NOTE = (
    "Size and aspect buckets filter ground-truth and predictions independently "
    "(COCO-style), then reuse evaluate(); a medium ground-truth box whose "
    "matching prediction is large is a false negative in the medium bucket and "
    "a false positive in the large bucket."
)


def _fmt_pct(value: Optional[float]) -> str:
    return "n/a" if value is None else f"{value * 100:.1f}%"


def _csv_num(value: Optional[float]) -> str:
    if value is None:
        return ""
    return repr(value)


def build_json_report(
    result: dict,
    generated_at: Optional[str] = None,
    extra_meta: Optional[dict] = None,
) -> dict:
    """Assemble the machine-readable geometry report. Deterministic given inputs."""
    config = result["config"]
    meta = {
        "artifact_type": ARTIFACT_TYPE,
        "schema_version": SCHEMA_VERSION,
        "iou_threshold": config["iou_threshold"],
        "classes": list(config["classes"]),
        "num_images": result["num_images"],
        "dataset_split": config["dataset_split"],
        "held_out_test_used": False,
        "coordinate_space": config["coordinate_space"],
        "size_buckets": config["size_buckets"],
        "aspect_buckets": config["aspect_buckets"],
        "strict": config["strict"],
        "independent_filtering": INDEPENDENT_FILTERING_NOTE,
    }
    if generated_at is not None:
        meta["generated_at"] = generated_at
    if extra_meta:
        meta.update(extra_meta)

    return {
        "meta": meta,
        "unstratified": result["unstratified"],
        "by_size": result["by_size"],
        "by_aspect": result["by_aspect"],
        "by_size_and_aspect": result["by_size_and_aspect"],
        "bucket_support": result["bucket_support"],
        "unknown_classes": result["unknown_classes"],
        "unmatched_image_ids": result["unmatched_image_ids"],
    }


def write_json_report(report: dict, path: Union[str, Path]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(report, f, indent=2, sort_keys=False)
        f.write("\n")


def _metric_row(dimension: str, bucket: str, cls: str, metrics: dict) -> dict:
    return {
        "dimension": dimension,
        "bucket": bucket,
        "class": cls,
        "support": metrics["support"],
        "tp": metrics["tp"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "precision": _csv_num(metrics["precision"]),
        "recall": _csv_num(metrics["recall"]),
        "f1": _csv_num(metrics["f1"]),
    }


def build_csv_rows(report: dict) -> list:
    """Flat rows: one per dimension × bucket × class, in a stable order."""
    classes = list(report["meta"]["classes"])
    rows = []
    for cls in classes:
        rows.append(_metric_row("unstratified", "all", cls, report["unstratified"]["per_class"][cls]))
    for bucket in SIZE_BUCKET_ORDER:
        for cls in classes:
            rows.append(_metric_row("size", bucket, cls, report["by_size"][bucket]["per_class"][cls]))
    for bucket in ASPECT_BUCKET_ORDER:
        for cls in classes:
            rows.append(
                _metric_row("aspect", bucket, cls, report["by_aspect"][bucket]["per_class"][cls])
            )
    for cls in classes:
        for size_bucket in SIZE_BUCKET_ORDER:
            for aspect_bucket in ASPECT_BUCKET_ORDER:
                key = f"{size_bucket}|{aspect_bucket}"
                rows.append(
                    _metric_row(
                        "size_and_aspect",
                        key,
                        cls,
                        report["by_size_and_aspect"][cls][key],
                    )
                )
    return rows


def write_csv_report(report: dict, path: Union[str, Path]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = build_csv_rows(report)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(CSV_FIELDNAMES))
        writer.writeheader()
        writer.writerows(rows)


def _metrics_table_row(label: str, metrics: dict) -> str:
    return (
        f"| {label} | {metrics['support']} | {metrics['tp']} | {metrics['fp']} | "
        f"{metrics['fn']} | {_fmt_pct(metrics['precision'])} | "
        f"{_fmt_pct(metrics['recall'])} | {_fmt_pct(metrics['f1'])} |"
    )


def _class_table(title: str, per_class: dict, classes: list) -> list:
    lines = [title, "", "| Class | Support | TP | FP | FN | Precision | Recall | F1 |", "|---|---|---|---|---|---|---|---|"]
    for cls in classes:
        lines.append(_metrics_table_row(cls, per_class[cls]))
    lines.append("")
    return lines


def build_markdown_report(report: dict, model_name: str = "candidate model") -> str:
    """Human-readable summary. Pole-focus section leads when pole is in classes."""
    meta = report["meta"]
    classes = list(meta["classes"])
    lines = []
    lines.append(f"# WalkBuddy Geometry Evaluation — {model_name}")
    lines.append("")
    lines.append(f"- Images evaluated: {meta['num_images']}")
    lines.append(f"- Dataset split: {meta['dataset_split']}")
    lines.append(f"- Held-out test used: {meta['held_out_test_used']}")
    lines.append(f"- IoU threshold: {meta['iou_threshold']}")
    lines.append(f"- Classes: {', '.join(classes)}")
    lines.append(f"- Coordinate space: {meta['coordinate_space']}")
    size_edges = meta["size_buckets"]
    lines.append(
        "- Size buckets (bbox area, px²; upper bound exclusive except large): "
        f"small {size_edges['small']}, medium {size_edges['medium']}, "
        f"large {size_edges['large']}"
    )
    aspect_edges = meta["aspect_buckets"]
    lines.append(
        "- Aspect buckets (height/width): "
        f"tall_thin ≥ {aspect_edges['tall_thin']['min_hw']}, "
        f"tall [{aspect_edges['tall']['min_hw']}, {aspect_edges['tall']['max_hw']}), "
        f"square_ish [{aspect_edges['square_ish']['min_hw']}, {aspect_edges['square_ish']['max_hw']}), "
        f"wide < {aspect_edges['wide']['max_hw']}"
    )
    if "generated_at" in meta:
        lines.append(f"- Generated: {meta['generated_at']}")
    lines.append("")
    lines.append(INDEPENDENT_FILTERING_NOTE)
    lines.append("")

    if "pole" in classes:
        pole_size_support = report["bucket_support"]["size"]["pole"]
        pole_aspect_support = report["bucket_support"]["aspect"]["pole"]
        lines.append("## Pole focus")
        lines.append("")
        lines.append("Ground-truth pole counts by bucket (before matching):")
        lines.append("")
        lines.append(
            "- Size: "
            + ", ".join(f"{bucket}={pole_size_support[bucket]}" for bucket in SIZE_BUCKET_ORDER)
        )
        lines.append(
            "- Aspect: "
            + ", ".join(f"{bucket}={pole_aspect_support[bucket]}" for bucket in ASPECT_BUCKET_ORDER)
        )
        lines.append("")
        lines.append("### Pole — by object size")
        lines.append("")
        lines.append("| Bucket | Support | TP | FP | FN | Precision | Recall | F1 |")
        lines.append("|---|---|---|---|---|---|---|---|")
        for bucket in SIZE_BUCKET_ORDER:
            lines.append(
                _metrics_table_row(bucket, report["by_size"][bucket]["per_class"]["pole"])
            )
        lines.append("")
        lines.append("### Pole — by aspect ratio")
        lines.append("")
        lines.append("| Bucket | Support | TP | FP | FN | Precision | Recall | F1 |")
        lines.append("|---|---|---|---|---|---|---|---|")
        for bucket in ASPECT_BUCKET_ORDER:
            lines.append(
                _metrics_table_row(bucket, report["by_aspect"][bucket]["per_class"]["pole"])
            )
        lines.append("")
        lines.append("### Pole — by size × aspect")
        lines.append("")
        lines.append("| Size | Aspect | Support | TP | FP | FN | Precision | Recall | F1 |")
        lines.append("|---|---|---|---|---|---|---|---|---|")
        for size_bucket in SIZE_BUCKET_ORDER:
            for aspect_bucket in ASPECT_BUCKET_ORDER:
                metrics = report["by_size_and_aspect"]["pole"][f"{size_bucket}|{aspect_bucket}"]
                lines.append(
                    f"| {size_bucket} | {aspect_bucket} | {metrics['support']} | "
                    f"{metrics['tp']} | {metrics['fp']} | {metrics['fn']} | "
                    f"{_fmt_pct(metrics['precision'])} | {_fmt_pct(metrics['recall'])} | "
                    f"{_fmt_pct(metrics['f1'])} |"
                )
        lines.append("")

    lines.append("## By object size (all requested classes)")
    lines.append("")
    for bucket in SIZE_BUCKET_ORDER:
        lines.extend(
            _class_table(
                f"### {bucket}",
                report["by_size"][bucket]["per_class"],
                classes,
            )
        )

    lines.append("## By aspect ratio (all requested classes)")
    lines.append("")
    for bucket in ASPECT_BUCKET_ORDER:
        lines.extend(
            _class_table(
                f"### {bucket}",
                report["by_aspect"][bucket]["per_class"],
                classes,
            )
        )

    lines.append("## Unstratified (all sizes and shapes)")
    lines.append("")
    overall = report["unstratified"]["overall"]
    lines.append("| Metric | Micro | Macro |")
    lines.append("|---|---|---|")
    lines.append(
        f"| Precision | {_fmt_pct(overall['micro']['precision'])} | {_fmt_pct(overall['macro']['precision'])} |"
    )
    lines.append(
        f"| Recall | {_fmt_pct(overall['micro']['recall'])} | {_fmt_pct(overall['macro']['recall'])} |"
    )
    lines.append(
        f"| F1 | {_fmt_pct(overall['micro']['f1'])} | {_fmt_pct(overall['macro']['f1'])} |"
    )
    lines.append(
        f"| TP / FP / FN | {overall['micro']['tp']} / {overall['micro']['fp']} / {overall['micro']['fn']} | — |"
    )
    lines.append("")
    lines.extend(_class_table("### Per-class", report["unstratified"]["per_class"], classes))

    return "\n".join(lines)


def write_markdown_report(markdown_text: str, path: Union[str, Path]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        f.write(markdown_text)


### 3.3 `evaluation/run_geometry_eval.py` — why and what

I wanted a CLI that is a sibling of `run_eval.py` and takes the **same JSON inputs**, plus a required `--split`.

I did **not** add `--model-path` in this version. The geometry CLI reads ground-truth JSON and predictions JSON only. That kept me from copying the Ultralytics loading loop out of `run_eval.py`. For Candidate 2, you export val predictions in this JSON schema first, then point this CLI at them.

`--split` is required (argparse `required=True`). Allowed values are `train` and `val` (`validation` and `eval` are aliases of val). `test` / held-out names, and paths whose components are `test` or contain `held-out` / `heldout`, are refused. There is **no override flag**. I was careful that a path under `tests/fixtures/...` is allowed — `tests` is not the same as a path component named `test`.

Thresholds can be overridden on the CLI (`--small-area-max`, `--tall-thin-min`, …) so the defaults are not hardcoded forever.

Actual file:

In [ ]:
"""CLI: size and aspect-ratio breakdown on top of evaluate().

Sibling of run_eval.py. Consumes the same ground-truth / predictions JSON
(image_id + boxes with class, bbox, and score on predictions). It does not
load a model and does not touch matching.py.

--split is required. The held-out test split is refused with no override.

Usage (synthetic fixtures, no trained model):

    python -m evaluation.run_geometry_eval \\
        --ground-truth tests/fixtures/eval/geometry/ground_truth.json \\
        --predictions tests/fixtures/eval/geometry/predictions.json \\
        --split val \\
        --out-dir reports/geometry_mock \\
        --model-name "mock (dev fixture)"

Usage (Candidate 2 on the validation split — never test):

    python -m evaluation.run_geometry_eval \\
        --ground-truth <val annotations JSON, pixel xyxy> \\
        --predictions <val predictions JSON, same schema> \\
        --split val \\
        --out-dir reports/candidate2_geometry \\
        --model-name "candidate_2"
"""

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

from .geometry import (
    AspectThresholds,
    HeldOutEvidenceError,
    SizeThresholds,
    evaluate_by_geometry,
    refuse_held_out_inputs,
)
from .geometry_report import (
    build_json_report,
    build_markdown_report,
    write_csv_report,
    write_json_report,
    write_markdown_report,
)
from .taxonomy import TAXONOMY_CLASSES


def _load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def run(
    ground_truth_path,
    predictions_path,
    out_dir,
    split,
    iou_threshold: float = 0.5,
    classes=None,
    model_name: str = "candidate model",
    strict: bool = True,
    size_thresholds=None,
    aspect_thresholds=None,
    deterministic_timestamp: bool = False,
):
    canonical_split = refuse_held_out_inputs(ground_truth_path, predictions_path, split)
    ground_truth = _load_json(ground_truth_path)
    predictions = _load_json(predictions_path)
    class_list = list(classes) if classes is not None else list(TAXONOMY_CLASSES)

    result = evaluate_by_geometry(
        ground_truth,
        predictions,
        split=canonical_split,
        classes=class_list,
        iou_threshold=iou_threshold,
        size_thresholds=size_thresholds,
        aspect_thresholds=aspect_thresholds,
        strict=strict,
    )

    generated_at = None if deterministic_timestamp else datetime.now(timezone.utc).isoformat()
    report = build_json_report(
        result,
        generated_at=generated_at,
        extra_meta={"model_name": model_name},
    )

    out_dir = Path(out_dir)
    write_json_report(report, out_dir / "geometry_eval_report.json")
    write_csv_report(report, out_dir / "geometry_eval_report.csv")
    write_markdown_report(
        build_markdown_report(report, model_name=model_name),
        out_dir / "geometry_eval_report.md",
    )
    return report


def main(argv=None):
    parser = argparse.ArgumentParser(
        description=(
            "WalkBuddy geometry evaluation: per-class precision/recall/F1 broken "
            "down by bbox size and aspect ratio. Reuses evaluate(); does not "
            "use the held-out test split."
        )
    )
    parser.add_argument("--ground-truth", required=True, help="Path to ground truth JSON")
    parser.add_argument("--predictions", required=True, help="Path to predictions JSON")
    parser.add_argument(
        "--split",
        required=True,
        help="Dataset split this JSON came from. Required. Allowed: train, val "
        "(aliases: validation, eval). test / held-out are refused.",
    )
    parser.add_argument("--out-dir", required=True, help="Directory to write report files into")
    parser.add_argument("--iou-threshold", type=float, default=0.5)
    parser.add_argument(
        "--classes",
        nargs="+",
        default=None,
        help="Subset of taxonomy classes to score (default: all eight). "
        "Example: --classes pole",
    )
    parser.add_argument("--model-name", default="candidate model")
    parser.add_argument(
        "--allow-unknown-classes",
        action="store_true",
        help="Surface out-of-taxonomy classes in the report instead of failing on them.",
    )
    parser.add_argument(
        "--small-area-max",
        type=float,
        default=None,
        help="Exclusive upper bound for the small size bucket (default: 1024 = 32^2).",
    )
    parser.add_argument(
        "--medium-area-max",
        type=float,
        default=None,
        help="Exclusive upper bound for the medium size bucket (default: 9216 = 96^2).",
    )
    parser.add_argument(
        "--tall-thin-min",
        type=float,
        default=None,
        help="Inclusive height/width lower bound for tall_thin (default: 3.0).",
    )
    parser.add_argument(
        "--tall-min",
        type=float,
        default=None,
        help="Inclusive height/width lower bound for tall (default: 1.5).",
    )
    parser.add_argument(
        "--square-min",
        type=float,
        default=None,
        help="Inclusive height/width lower bound for square_ish (default: 2/3).",
    )
    args = parser.parse_args(argv)

    size_kwargs = {}
    if args.small_area_max is not None:
        size_kwargs["small_max"] = args.small_area_max
    if args.medium_area_max is not None:
        size_kwargs["medium_max"] = args.medium_area_max
    aspect_kwargs = {}
    if args.tall_thin_min is not None:
        aspect_kwargs["tall_thin_min"] = args.tall_thin_min
    if args.tall_min is not None:
        aspect_kwargs["tall_min"] = args.tall_min
    if args.square_min is not None:
        aspect_kwargs["square_min"] = args.square_min

    try:
        report = run(
            ground_truth_path=args.ground_truth,
            predictions_path=args.predictions,
            out_dir=args.out_dir,
            split=args.split,
            iou_threshold=args.iou_threshold,
            classes=args.classes,
            model_name=args.model_name,
            strict=not args.allow_unknown_classes,
            size_thresholds=SizeThresholds(**size_kwargs) if size_kwargs else None,
            aspect_thresholds=AspectThresholds(**aspect_kwargs) if aspect_kwargs else None,
        )
    except HeldOutEvidenceError as exc:
        print(f"Geometry evaluation refused: {exc}", file=sys.stderr)
        return 1
    except ValueError as exc:
        print(f"Geometry evaluation failed: {exc}", file=sys.stderr)
        return 1

    print(f"Geometry evaluation complete: {args.out_dir}")
    print(f"Split: {report['meta']['dataset_split']}")
    print(f"Held-out test used: {report['meta']['held_out_test_used']}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### 3.4 `tests/test_evaluation_geometry.py` — why and what

The task asked for automated tests, and the PR evidence is supposed to be **synthetic fixtures + passing tests**, not a re-score of held-out Candidate 1. So I built a small fixture set under `tests/fixtures/eval/geometry/` with known boxes:

- small square pole 10×10
- medium tall-thin pole 20×200
- large wide table 200×80
- large tall-thin pole 40×400
- medium wide pole 200×40
- a **cross-size** pair: ground-truth pole 30×30 (small) vs prediction 40×40 (medium), IoU ≈ 0.56 so they *would* match if evaluated together

The tests I care about most:

- **bucket assignment** — COCO boundaries (1024 / 9216) and aspect boundaries (3.0 / 1.5 / 2/3)
- **size / aspect breakdown** — pole small recall 0.5, medium precision 2/3, empty `tall` bucket is `n/a` not a crash
- **independent filtering** — that cross-size pair is a TP unstratified, FN in small, FP in medium
- **`--classes pole`** — other taxonomy classes (table) are excluded without being treated as unknown
- **held-out guard** — `split="test"` raises; a `held-out/` path raises even with `--split val`; CLI exit code 1; our own `tests/fixtures` path is *not* blocked
- **determinism** — two runs with a fixed timestamp produce identical reports
- **JSON / CSV / Markdown** — CLI writes all three; Markdown leads with Pole focus when pole is requested

I also checked that the *old* eval fixture pole still lands in `medium` + `tall_thin`, so I did not silently change how existing boxes are measured.

Actual test file:

In [ ]:
import csv
import json
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))  # ML_side, so `evaluation` is importable

from evaluation.geometry import (
    DEFAULT_MEDIUM_AREA_MAX,
    DEFAULT_SMALL_AREA_MAX,
    AspectThresholds,
    HeldOutEvidenceError,
    SizeThresholds,
    aspect_bucket_for_hw,
    bbox_area,
    bbox_aspect_hw,
    box_aspect_bucket,
    box_size_bucket,
    canonical_split,
    evaluate_by_geometry,
    filter_records,
    path_is_held_out,
    refuse_held_out_inputs,
    refuse_held_out_path,
    size_bucket_for_area,
)
from evaluation.geometry_report import (
    build_csv_rows,
    build_json_report,
    build_markdown_report,
)
from evaluation.metrics import evaluate
from evaluation.run_geometry_eval import run
from evaluation.taxonomy import TAXONOMY_CLASSES

FIXTURE_DIR = Path(__file__).parent / "fixtures" / "eval" / "geometry"
GT_PATH = FIXTURE_DIR / "ground_truth.json"
PRED_PATH = FIXTURE_DIR / "predictions.json"
SMALL_GT_PATH = Path(__file__).parent / "fixtures" / "eval" / "ground_truth_small.json"
SMALL_PRED_PATH = Path(__file__).parent / "fixtures" / "eval" / "predictions_small.json"


def _load_geometry_fixtures():
    with open(GT_PATH) as f:
        gt = json.load(f)
    with open(PRED_PATH) as f:
        preds = json.load(f)
    return gt, preds


# ---- unit: area / aspect / buckets ----

def test_bbox_area_and_aspect_of_known_boxes():
    assert bbox_area([0, 0, 10, 10]) == pytest.approx(100)
    assert bbox_aspect_hw([0, 0, 10, 10]) == pytest.approx(1.0)
    assert bbox_area([10, 10, 30, 210]) == pytest.approx(4000)
    assert bbox_aspect_hw([10, 10, 30, 210]) == pytest.approx(10.0)
    assert bbox_area([0, 0, 200, 80]) == pytest.approx(16000)
    assert bbox_aspect_hw([0, 0, 200, 80]) == pytest.approx(0.4)
    assert bbox_area([0, 0, 0, 10]) == 0.0
    assert bbox_aspect_hw([0, 0, 0, 10]) is None


def test_coco_size_bucket_boundaries():
    thresholds = SizeThresholds()
    assert thresholds.small_max == DEFAULT_SMALL_AREA_MAX
    assert thresholds.medium_max == DEFAULT_MEDIUM_AREA_MAX
    assert size_bucket_for_area(0, thresholds) == "invalid"
    assert size_bucket_for_area(1023, thresholds) == "small"
    assert size_bucket_for_area(1024, thresholds) == "medium"
    assert size_bucket_for_area(9215, thresholds) == "medium"
    assert size_bucket_for_area(9216, thresholds) == "large"


def test_aspect_bucket_boundaries():
    thresholds = AspectThresholds()
    assert aspect_bucket_for_hw(None, thresholds) == "invalid"
    assert aspect_bucket_for_hw(3.0, thresholds) == "tall_thin"
    assert aspect_bucket_for_hw(2.999, thresholds) == "tall"
    assert aspect_bucket_for_hw(1.5, thresholds) == "tall"
    assert aspect_bucket_for_hw(1.499, thresholds) == "square_ish"
    assert aspect_bucket_for_hw(2.0 / 3.0, thresholds) == "square_ish"
    assert aspect_bucket_for_hw((2.0 / 3.0) - 1e-9, thresholds) == "wide"


def test_size_thresholds_are_configurable():
    tight = SizeThresholds(small_max=50, medium_max=200)
    assert size_bucket_for_area(49, tight) == "small"
    assert size_bucket_for_area(50, tight) == "medium"
    assert size_bucket_for_area(200, tight) == "large"


def test_filter_records_keeps_image_ids_and_extra_fields():
    records = [
        {
            "image_id": "imgA",
            "image_path": "/tmp/imgA.jpg",
            "boxes": [
                {"class": "pole", "bbox": [0, 0, 10, 10]},
                {"class": "pole", "bbox": [0, 0, 40, 400]},
            ],
        },
        {"image_id": "imgB", "boxes": []},
    ]
    thresholds = SizeThresholds()
    filtered = filter_records(
        records, lambda box: box_size_bucket(box, thresholds) == "small"
    )
    assert [rec["image_id"] for rec in filtered] == ["imgA", "imgB"]
    assert filtered[0]["image_path"] == "/tmp/imgA.jpg"
    assert len(filtered[0]["boxes"]) == 1
    assert filtered[0]["boxes"][0]["bbox"] == [0, 0, 10, 10]
    assert filtered[1]["boxes"] == []
    assert len(records[0]["boxes"]) == 2  # original not mutated


# ---- split / held-out guard ----

def test_canonical_split_aliases_and_required():
    assert canonical_split("train") == "train"
    assert canonical_split("val") == "val"
    assert canonical_split("validation") == "val"
    assert canonical_split("eval") == "val"
    assert canonical_split("VAL") == "val"
    with pytest.raises(HeldOutEvidenceError, match="required"):
        canonical_split("")
    with pytest.raises(HeldOutEvidenceError, match="required"):
        canonical_split(None)
    with pytest.raises(ValueError, match="Unknown dataset split"):
        canonical_split("holdout")


@pytest.mark.parametrize("split", ["test", "TEST", "heldout", "held-out", "held_out", "held-out-test"])
def test_canonical_split_refuses_held_out_names(split):
    with pytest.raises(HeldOutEvidenceError, match="held-out"):
        canonical_split(split)


def test_path_is_held_out_distinguishes_tests_from_test():
    assert path_is_held_out("dataset/test/labels.json") is True
    assert path_is_held_out("dataset/test.json") is True
    assert path_is_held_out("walkbuddy-heldout-dataset.json") is True
    assert path_is_held_out("candidates/foo-heldout-test-corrected/summary.json") is True
    assert path_is_held_out(GT_PATH) is False
    assert path_is_held_out("tests/fixtures/eval/geometry/ground_truth.json") is False


def test_refuse_held_out_path_and_inputs():
    with pytest.raises(HeldOutEvidenceError):
        refuse_held_out_path("data/held-out/gt.json")
    with pytest.raises(HeldOutEvidenceError):
        refuse_held_out_inputs("data/test/gt.json", "data/val/preds.json", "val")
    with pytest.raises(HeldOutEvidenceError):
        refuse_held_out_inputs(GT_PATH, PRED_PATH, "test")
    assert refuse_held_out_inputs(GT_PATH, PRED_PATH, "validation") == "val"


def test_evaluate_by_geometry_requires_split_and_records_held_out_false():
    gt, preds = _load_geometry_fixtures()
    with pytest.raises(TypeError):
        evaluate_by_geometry(gt, preds)
    result = evaluate_by_geometry(gt, preds, split="val")
    assert result["config"]["held_out_test_used"] is False
    assert result["config"]["dataset_split"] == "val"


def test_evaluate_by_geometry_refuses_test_split():
    gt, preds = _load_geometry_fixtures()
    with pytest.raises(HeldOutEvidenceError):
        evaluate_by_geometry(gt, preds, split="test")


def test_normalized_boxes_are_rejected():
    gt = [{"image_id": "n", "boxes": [{"class": "pole", "bbox": [0.1, 0.1, 0.2, 0.4]}]}]
    preds = [{"image_id": "n", "boxes": [{"class": "pole", "bbox": [0.1, 0.1, 0.2, 0.4], "score": 0.9}]}]
    with pytest.raises(ValueError, match="pixel xyxy"):
        evaluate_by_geometry(gt, preds, split="val")


# ---- fixture geometry assignments ----

def test_geometry_fixture_bucket_assignments():
    thresholds_s = SizeThresholds()
    thresholds_a = AspectThresholds()
    gt, preds = _load_geometry_fixtures()
    by_id = {rec["image_id"]: rec["boxes"][0] for rec in gt}

    assert box_size_bucket(by_id["img_small_square_pole"], thresholds_s) == "small"
    assert box_aspect_bucket(by_id["img_small_square_pole"], thresholds_a) == "square_ish"

    assert box_size_bucket(by_id["img_medium_tall_thin_pole"], thresholds_s) == "medium"
    assert box_aspect_bucket(by_id["img_medium_tall_thin_pole"], thresholds_a) == "tall_thin"

    assert box_size_bucket(by_id["img_large_wide_table"], thresholds_s) == "large"
    assert box_aspect_bucket(by_id["img_large_wide_table"], thresholds_a) == "wide"

    assert box_size_bucket(by_id["img_cross_size_pole"], thresholds_s) == "small"
    assert box_aspect_bucket(by_id["img_cross_size_pole"], thresholds_a) == "square_ish"
    cross_pred = [rec for rec in preds if rec["image_id"] == "img_cross_size_pole"][0]["boxes"][0]
    assert box_size_bucket(cross_pred, thresholds_s) == "medium"
    assert box_aspect_bucket(cross_pred, thresholds_a) == "square_ish"

    assert box_size_bucket(by_id["img_large_tall_thin_pole"], thresholds_s) == "large"
    assert box_aspect_bucket(by_id["img_large_tall_thin_pole"], thresholds_a) == "tall_thin"

    assert box_size_bucket(by_id["img_medium_wide_pole"], thresholds_s) == "medium"
    assert box_aspect_bucket(by_id["img_medium_wide_pole"], thresholds_a) == "wide"


# ---- evaluate_by_geometry scoring ----

def test_evaluate_by_geometry_is_deterministic():
    gt, preds = _load_geometry_fixtures()
    a = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES, iou_threshold=0.5)
    b = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES, iou_threshold=0.5)
    assert a == b


def test_unstratified_matches_plain_evaluate():
    gt, preds = _load_geometry_fixtures()
    plain = evaluate(gt, preds, classes=TAXONOMY_CLASSES, iou_threshold=0.5)
    geo = evaluate_by_geometry(gt, preds, split="train", classes=TAXONOMY_CLASSES, iou_threshold=0.5)
    assert geo["unstratified"]["per_class"] == plain["per_class"]
    assert geo["unstratified"]["overall"] == plain["overall"]
    assert geo["num_images"] == 6
    # Unstratified, the cross-size pair still matches (IoU 0.5625), so pole is 5 TPs.
    assert geo["unstratified"]["per_class"]["pole"]["tp"] == 5
    assert geo["unstratified"]["per_class"]["pole"]["fp"] == 0
    assert geo["unstratified"]["per_class"]["pole"]["fn"] == 0
    assert geo["unstratified"]["per_class"]["table"]["tp"] == 1


def test_size_breakdown_for_pole_and_table():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES)

    pole_small = result["by_size"]["small"]["per_class"]["pole"]
    assert pole_small["support"] == 2  # 10x10 and 30x30
    assert pole_small["tp"] == 1
    assert pole_small["fn"] == 1  # 30x30 GT, prediction landed in medium
    assert pole_small["fp"] == 0
    assert pole_small["precision"] == pytest.approx(1.0)
    assert pole_small["recall"] == pytest.approx(0.5)

    pole_medium = result["by_size"]["medium"]["per_class"]["pole"]
    assert pole_medium["support"] == 2  # 20x200 and 200x40
    assert pole_medium["tp"] == 2
    assert pole_medium["fp"] == 1  # 40x40 prediction from the cross-size image
    assert pole_medium["fn"] == 0
    assert pole_medium["precision"] == pytest.approx(2 / 3)
    assert pole_medium["recall"] == pytest.approx(1.0)

    pole_large = result["by_size"]["large"]["per_class"]["pole"]
    assert pole_large["support"] == 1
    assert pole_large["tp"] == 1
    assert pole_large["fp"] == 0
    assert pole_large["fn"] == 0

    table_large = result["by_size"]["large"]["per_class"]["table"]
    assert table_large["support"] == 1
    assert table_large["tp"] == 1
    assert result["by_size"]["small"]["per_class"]["table"]["support"] == 0


def test_aspect_breakdown_including_empty_tall_bucket():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES)

    pole_thin = result["by_aspect"]["tall_thin"]["per_class"]["pole"]
    assert pole_thin["support"] == 2
    assert pole_thin["tp"] == 2
    assert pole_thin["fp"] == 0
    assert pole_thin["fn"] == 0

    pole_square = result["by_aspect"]["square_ish"]["per_class"]["pole"]
    # Both the 10x10 pair and the 30x30 vs 40x40 pair are square_ish, and the
    # latter still matches on IoU when aspect (not size) is the slice.
    assert pole_square["support"] == 2
    assert pole_square["tp"] == 2
    assert pole_square["fp"] == 0
    assert pole_square["fn"] == 0

    pole_wide = result["by_aspect"]["wide"]["per_class"]["pole"]
    assert pole_wide["support"] == 1
    assert pole_wide["tp"] == 1

    pole_tall = result["by_aspect"]["tall"]["per_class"]["pole"]
    assert pole_tall["support"] == 0
    assert pole_tall["tp"] == 0
    assert pole_tall["fp"] == 0
    assert pole_tall["fn"] == 0
    assert pole_tall["precision"] is None
    assert pole_tall["recall"] is None
    assert pole_tall["f1"] is None

    table_wide = result["by_aspect"]["wide"]["per_class"]["table"]
    assert table_wide["support"] == 1
    assert table_wide["tp"] == 1


def test_cross_size_pair_is_fn_and_fp_under_independent_filtering():
    """The 30x30 GT / 40x40 pred pair is a TP unstratified (IoU 0.5625) but
    splits into FN(small) + FP(medium) when size-filtered independently."""
    gt = [{"image_id": "cross", "boxes": [{"class": "pole", "bbox": [0, 0, 30, 30]}]}]
    preds = [{"image_id": "cross", "boxes": [{"class": "pole", "bbox": [0, 0, 40, 40], "score": 0.8}]}]
    result = evaluate_by_geometry(gt, preds, split="val", classes=["pole"])
    assert result["unstratified"]["per_class"]["pole"]["tp"] == 1
    assert result["by_size"]["small"]["per_class"]["pole"]["fn"] == 1
    assert result["by_size"]["small"]["per_class"]["pole"]["tp"] == 0
    assert result["by_size"]["medium"]["per_class"]["pole"]["fp"] == 1
    assert result["by_size"]["medium"]["per_class"]["pole"]["tp"] == 0
    assert result["by_size"]["large"]["per_class"]["pole"]["support"] == 0


def test_classes_pole_excludes_other_taxonomy_classes():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=["pole"])
    assert list(result["unstratified"]["per_class"]) == ["pole"]
    assert "table" not in result["by_size"]["large"]["per_class"]
    assert "table" not in result["by_size_and_aspect"]
    assert result["unstratified"]["per_class"]["pole"]["tp"] == 5


def test_size_and_aspect_cross_tab_for_pole():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=["pole"])
    crossed = result["by_size_and_aspect"]["pole"]
    assert crossed["small|square_ish"]["support"] == 2
    assert crossed["small|square_ish"]["tp"] == 1
    assert crossed["small|square_ish"]["fn"] == 1
    assert crossed["medium|tall_thin"]["tp"] == 1
    assert crossed["medium|wide"]["tp"] == 1
    assert crossed["large|tall_thin"]["tp"] == 1
    assert crossed["small|tall_thin"]["support"] == 0
    assert crossed["small|tall_thin"]["precision"] is None


def test_bucket_support_counts_ground_truth_only():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES)
    pole_size = result["bucket_support"]["size"]["pole"]
    assert pole_size["small"] == 2
    assert pole_size["medium"] == 2
    assert pole_size["large"] == 1
    assert pole_size["invalid"] == 0
    pole_aspect = result["bucket_support"]["aspect"]["pole"]
    assert pole_aspect["tall_thin"] == 2
    assert pole_aspect["tall"] == 0
    assert pole_aspect["square_ish"] == 2
    assert pole_aspect["wide"] == 1


def test_existing_eval_fixture_pole_is_medium_tall_thin():
    with open(SMALL_GT_PATH) as f:
        gt = json.load(f)
    with open(SMALL_PRED_PATH) as f:
        preds = json.load(f)
    result = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES)
    pole_box = [b for rec in gt for b in rec["boxes"] if b["class"] == "pole"][0]
    assert box_size_bucket(pole_box, SizeThresholds()) == "medium"
    assert box_aspect_bucket(pole_box, AspectThresholds()) == "tall_thin"
    assert result["by_size"]["medium"]["per_class"]["pole"]["tp"] == 1
    assert result["by_aspect"]["tall_thin"]["per_class"]["pole"]["tp"] == 1


# ---- reports ----

def test_json_report_records_guard_metadata_and_is_deterministic():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="eval", classes=TAXONOMY_CLASSES)
    report_a = build_json_report(result, extra_meta={"model_name": "mock"})
    report_b = build_json_report(result, extra_meta={"model_name": "mock"})
    assert report_a == report_b
    assert report_a["meta"]["artifact_type"] == "geometry_breakdown"
    assert report_a["meta"]["dataset_split"] == "val"
    assert report_a["meta"]["held_out_test_used"] is False
    assert report_a["meta"]["coordinate_space"] == "pixel_xyxy"
    assert report_a["meta"]["size_buckets"]["small"] == [0, 1024]
    assert report_a["meta"]["size_buckets"]["medium"] == [1024, 9216]
    assert report_a["meta"]["size_buckets"]["large"] == [9216, None]
    assert "generated_at" not in report_a["meta"]


def test_csv_rows_cover_each_dimension_bucket_class():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=["pole", "table"])
    report = build_json_report(result)
    rows = build_csv_rows(report)
    # 2 classes * (1 unstratified + 3 size + 4 aspect + 12 size_and_aspect) = 40
    assert len(rows) == 2 * (1 + 3 + 4 + 12)
    pole_small = [
        row for row in rows if row["dimension"] == "size" and row["bucket"] == "small" and row["class"] == "pole"
    ][0]
    assert pole_small["support"] == 2
    assert pole_small["tp"] == 1
    assert pole_small["fn"] == 1
    empty = [
        row
        for row in rows
        if row["dimension"] == "aspect" and row["bucket"] == "tall" and row["class"] == "pole"
    ][0]
    assert empty["precision"] == ""
    assert empty["recall"] == ""


def test_markdown_leads_with_pole_focus_and_notes_independent_filtering():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=TAXONOMY_CLASSES)
    report = build_json_report(result, extra_meta={"model_name": "mock"})
    md = build_markdown_report(report, model_name="mock")
    assert md.index("## Pole focus") < md.index("## By object size")
    assert "Pole — by object size" in md
    assert "Pole — by aspect ratio" in md
    assert "Pole — by size × aspect" in md
    assert "independently" in md
    assert "Held-out test used: False" in md
    assert "tall_thin" in md


def test_markdown_omits_pole_focus_when_pole_not_requested():
    gt, preds = _load_geometry_fixtures()
    result = evaluate_by_geometry(gt, preds, split="val", classes=["table"])
    report = build_json_report(result)
    md = build_markdown_report(report, model_name="table only")
    assert "## Pole focus" not in md
    assert "### table" in md or "| table |" in md


# ---- CLI ----

def test_run_writes_json_csv_and_markdown(tmp_path):
    out_dir = tmp_path / "geo"
    report = run(
        ground_truth_path=GT_PATH,
        predictions_path=PRED_PATH,
        out_dir=out_dir,
        split="val",
        model_name="mock (dev fixture)",
        deterministic_timestamp=True,
    )
    json_path = out_dir / "geometry_eval_report.json"
    csv_path = out_dir / "geometry_eval_report.csv"
    md_path = out_dir / "geometry_eval_report.md"
    assert json_path.exists()
    assert csv_path.exists()
    assert md_path.exists()
    with open(json_path) as f:
        saved = json.load(f)
    assert saved == report
    assert saved["meta"]["held_out_test_used"] is False
    with open(csv_path, newline="") as f:
        rows = list(csv.DictReader(f))
    assert rows[0]["dimension"] == "unstratified"
    md = md_path.read_text()
    assert "## Pole focus" in md


def test_run_is_deterministic_across_repeated_calls(tmp_path):
    report_a = run(
        ground_truth_path=GT_PATH,
        predictions_path=PRED_PATH,
        out_dir=tmp_path / "a",
        split="val",
        deterministic_timestamp=True,
    )
    report_b = run(
        ground_truth_path=GT_PATH,
        predictions_path=PRED_PATH,
        out_dir=tmp_path / "b",
        split="val",
        deterministic_timestamp=True,
    )
    assert report_a == report_b


def test_run_classes_pole_only(tmp_path):
    report = run(
        ground_truth_path=GT_PATH,
        predictions_path=PRED_PATH,
        out_dir=tmp_path / "pole_only",
        split="train",
        classes=["pole"],
        deterministic_timestamp=True,
    )
    assert report["meta"]["classes"] == ["pole"]
    assert report["meta"]["dataset_split"] == "train"
    assert "table" not in report["unstratified"]["per_class"]


def test_run_rejects_test_split(tmp_path):
    with pytest.raises(HeldOutEvidenceError):
        run(
            ground_truth_path=GT_PATH,
            predictions_path=PRED_PATH,
            out_dir=tmp_path / "nope",
            split="test",
        )


def test_run_rejects_held_out_path_even_with_val_split(tmp_path):
    heldout_gt = tmp_path / "held-out" / "gt.json"
    heldout_gt.parent.mkdir()
    heldout_gt.write_text(GT_PATH.read_text())
    with pytest.raises(HeldOutEvidenceError):
        run(
            ground_truth_path=heldout_gt,
            predictions_path=PRED_PATH,
            out_dir=tmp_path / "out",
            split="val",
        )


def test_cli_requires_split(tmp_path):
    from evaluation.run_geometry_eval import main

    with pytest.raises(SystemExit):
        main(
            [
                "--ground-truth",
                str(GT_PATH),
                "--predictions",
                str(PRED_PATH),
                "--out-dir",
                str(tmp_path / "cli"),
            ]
        )


def test_cli_split_test_returns_nonzero(tmp_path, capsys):
    from evaluation.run_geometry_eval import main

    code = main(
        [
            "--ground-truth",
            str(GT_PATH),
            "--predictions",
            str(PRED_PATH),
            "--split",
            "test",
            "--out-dir",
            str(tmp_path / "cli_test"),
        ]
    )
    assert code == 1
    err = capsys.readouterr().err
    assert "held-out" in err.lower()


## 4. The design decisions I made

**Size cuts: COCO 32² / 96², configurable.** I did not have an in-repo histogram of WalkBuddy pole areas, so I did not invent dataset-specific thresholds. COCO's small < 32², medium < 96², large otherwise is the usual object-detection split, it matches training `imgsz` 640, and the CLI can override it later if val-set stats justify different cuts. Inputs have to be **pixel xyxy** — the same unit `run_eval` already uses.

**Aspect cuts: height / width, tall_thin at 3.0 not 4.0.** I used h/w so a lamp-post is a large number instead of a fraction. Defaults:

- `tall_thin`: h/w ≥ 3.0 (the existing fixture pole is 18.5; a 20×200 pole is 10)
- `tall`: 1.5 ≤ h/w < 3.0
- `square_ish`: 2/3 ≤ h/w < 1.5
- `wide`: h/w < 2/3

I considered raising `tall_thin` to 4.0 so doors would not share the bucket with poles. I kept **3.0** because this task is about capturing poles, not about a door-vs-pole confusion matrix. The existing fixture door (h/w 3.5) would land in `tall_thin` too; that is accepted.

**Matching: independent filter-then-`evaluate()`, COCO-style.** I filter GT and predictions separately, then call the existing `evaluate()`. That means a medium ground-truth box whose matching prediction is large is a **false negative in medium** and a **false positive in large**, even if unstratified they were a true positive. I documented that in the Markdown report on purpose. The alternative (match globally, then attribute) would have meant reimplementing matching, which I was asked not to do.

**Held-out test: hard refuse, no override, `--split` required.** I do not want this tool to be a quiet way to peek at test. `held_out_test_used` is always `false` in the JSON. Path heuristic: a folder named `test` is blocked; `tests/fixtures` is not.

## 5. How I verified it

I ran the geometry tests first, then the full `ML_side/tests` suite. I also ran the CLI on the synthetic fixtures the way a reviewer would.

**Geometry tests:** 38 passed.

```
ML_side/tests/test_evaluation_geometry.py::test_cli_split_test_returns_nonzero PASSED [100%]
============================== 38 passed in 0.12s ==============================
```

**Full ML_side suite:** 662 passed, 0 failed.

**CLI smoke run** (val split, synthetic fixtures) wrote three files:

- `reports/geometry_mock/geometry_eval_report.json`
- `reports/geometry_mock/geometry_eval_report.csv`
- `reports/geometry_mock/geometry_eval_report.md`

Header from that run: `Split: val`, `Held-out test used: False`.

The Pole focus tables from the **real Markdown** I generated are below. This is the point of the tool: unstratified pole looks perfect (5/5) on this fixture, but **small-pole recall is 50%** because the 30×30 vs 40×40 pair is split across size buckets. That is the independent-filtering behaviour working, and it is also what you would hope to see on a real val set if small poles were the failure mode.

```
## Pole focus

Ground-truth pole counts by bucket (before matching):

- Size: small=2, medium=2, large=1
- Aspect: tall_thin=2, tall=0, square_ish=2, wide=1

### Pole — by object size

| Bucket | Support | TP | FP | FN | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|
| small | 2 | 1 | 0 | 1 | 100.0% | 50.0% | 66.7% |
| medium | 2 | 2 | 1 | 0 | 66.7% | 100.0% | 80.0% |
| large | 1 | 1 | 0 | 0 | 100.0% | 100.0% | 100.0% |

### Pole — by aspect ratio

| Bucket | Support | TP | FP | FN | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|
| tall_thin | 2 | 2 | 0 | 0 | 100.0% | 100.0% | 100.0% |
| tall | 0 | 0 | 0 | 0 | n/a | n/a | n/a |
| square_ish | 2 | 2 | 0 | 0 | 100.0% | 100.0% | 100.0% |
| wide | 1 | 1 | 0 | 0 | 100.0% | 100.0% | 100.0% |
```

Unstratified on the same run was 6 TP / 0 FP / 0 FN (5 poles + 1 table). I did **not** re-score Candidate 1's held-out test set.

The next cell re-runs the fixture scoring and shows the CLI refusing `--split test`.

In [ ]:
import json
# Re-run the geometry tool on the synthetic fixtures and refuse --split test.
# Works if the notebook kernel's cwd is the repo root, ML_side, or ML_side/docs.

from pathlib import Path
import sys

here = Path.cwd()
candidates = [here, here.parent, here / "ML_side", here.parent / "ML_side"]
ml_side = next(
    (c.resolve() for c in candidates if (c / "evaluation" / "geometry.py").exists()),
    None,
)
assert ml_side is not None, f"Could not find ML_side from {here}"
sys.path.insert(0, str(ml_side))

from evaluation.geometry import HeldOutEvidenceError, evaluate_by_geometry
from evaluation.run_geometry_eval import main

gt = json.loads((ml_side / "tests/fixtures/eval/geometry/ground_truth.json").read_text())
preds = json.loads((ml_side / "tests/fixtures/eval/geometry/predictions.json").read_text())

result = evaluate_by_geometry(gt, preds, split="val")
pole_small = result["by_size"]["small"]["per_class"]["pole"]
print("split:", result["config"]["dataset_split"])
print("held_out_test_used:", result["config"]["held_out_test_used"])
print("pole small support/tp/fn:", pole_small["support"], pole_small["tp"], pole_small["fn"])
print("pole small recall:", pole_small["recall"])

print("\nCLI --split test:")
code = main([
    "--ground-truth", str(ml_side / "tests/fixtures/eval/geometry/ground_truth.json"),
    "--predictions", str(ml_side / "tests/fixtures/eval/geometry/predictions.json"),
    "--split", "test",
    "--out-dir", str(ml_side / "reports" / "_geometry_writeup_should_not_exist"),
])
print("exit code:", code)


## 6. How to rerun for Candidate 2

Use **val** only. Do not pass `--split test`, do not point at held-out folders, and do not use `evaluate_current_model.py --split test` artifacts as input.

Predictions must be the same JSON schema as `evaluate()` / `run_eval` (pixel xyxy), not Ultralytics' labelled-validation dump.

```bash
cd ML_side
python -m evaluation.run_geometry_eval \
  --ground-truth <path to val-split annotations JSON> \
  --predictions <path to val-split predictions JSON> \
  --split val \
  --out-dir reports/candidate2_geometry \
  --model-name "candidate_2"
```

Optional: `--classes pole` if you only want pole rows. Later candidates are the same command with a new `--model-name` and new prediction JSON.

There is a longer version of this (including the optional `run_eval` overall report) in `ML_side/docs/EVALUATION_PIPELINE.md`.

## 7. What I learned / why this matters

The main thing I learned is that I did not need a second matcher. Once `evaluate()` already did IoU, precision, recall, and F1, the useful work was **slicing the same boxes** and refusing to lie about which split I scored.

That matters for poles specifically because one overall pole F1 cannot tell you whether the model is failing on distant thin posts, nearby thick ones, or something else. Candidate 1's held-out pole precision was very low in a report I was told **not** to tune against — this tool is how we can look at that kind of weakness on **val** for Candidate 2, with buckets a reviewer can reproduce.

It also stays reusable: `--classes` is just a list. If the next weak class is `person` or `bicycle`, the same CLI and the same reports work. The Markdown only special-cases pole as a leading section, not as a hardcoded-only class.

Limits I am not hiding: this is still single-IoU-threshold P/R/F1 (same as the existing eval, not COCO mAP), independent filtering can double-count a mismatch as FN+FP across buckets, and I have not run it on a real Candidate 2 val dump yet — only on synthetic fixtures and the existing eval fixture pole.